**🚕 Phân tích yếu tố ảnh hưởng đến GIÁ & SURGE**
**Dự án Pricing Research (XanhSM) — Tuần 1, Mục i**

Dữ liệu: `dataset_clean.parquet` — **637.976 chuyến** đã làm sạch (bộ FULL).
> Chạy `00_clean_full_dataset.ipynb` trước nếu chưa có file này.

---
**⚠️ Hai nguyên tắc xuyên suốt notebook**

**1. Giá thô bị `quãng đường × loại dịch vụ` chi phối** → hiệu ứng thời tiết/thời gian/vị trí bị che lấp.
Vì vậy luôn phân tích song song **`price`** và **`price_per_mile`** (giá đã chuẩn hoá quãng đường).

**2. Uber KHÔNG có dữ liệu surge** (330.568 dòng đều = 1.0 — hạn chế của API).
Vì vậy **mọi phân tích về surge chỉ dùng Lyft** (`dfL`), nếu không sẽ bị pha loãng bằng số 0 giả.

> ▶️ **Run All** để chạy toàn bộ.


**0. Nạp dữ liệu đã làm sạch**

In [ ]:
%matplotlib inline
import math, warnings, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80); pd.set_option("display.width", 200)

BLUE, ORANGE, GREEN, RED, PURPLE, MUT = "#0072B2","#E69F00","#009E73","#D55E00","#CC79A7","#666666"
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white",
    "axes.spines.top":False,"axes.spines.right":False,"axes.grid":True,
    "grid.color":"#ECECEC","grid.linewidth":0.8,"font.size":10})

NAME="dataset_clean.parquet"
p = next((c/NAME for c in [Path("../data"),Path("data"),Path(".")] if (c/NAME).exists()), None)
if p is None: raise FileNotFoundError("Chua co file sach - hay chay 00_clean_full_dataset.ipynb truoc")
t0=time.time(); df = pd.read_parquet(p)
print(f"Nap {len(df):,} chuyen x {df.shape[1]} cot trong {time.time()-t0:.1f}s")

# dfL = chi Lyft -> dung cho MOI phan tich ve surge
dfL = df[df.cab_type=="Lyft"].copy()
print(f"df  (ca 2 hang, phan tich GIA)  : {len(df):,} chuyen")
print(f"dfL (chi Lyft, phan tich SURGE) : {len(dfL):,} chuyen | {dfL.is_surge.sum():,} cuoc co surge "
      f"({dfL.is_surge.mean()*100:.2f}%)")

**1. Tổng quan nhanh**

In [ ]:
print("="*62)
print(f"Chuyen        : {len(df):,}")
print(f"Thoi gian     : {df.date_local.min().date()} -> {df.date_local.max().date()} ({df.date_local.nunique()} ngay)")
print(f"Khu vuc       : {df.source.nunique()} | Dich vu: {df.name.nunique()} | Hang: {df.cab_type.nunique()}")
print(f"Gia           : {df.price.min():.2f} - {df.price.max():.2f} USD (TB {df.price.mean():.2f}, median {df.price.median():.2f})")
print(f"Quang duong   : {df.distance.min():.2f} - {df.distance.max():.2f} dam (TB {df.distance.mean():.2f})")
print(f"Gia/dam       : median {df.price_per_mile.median():.2f} USD/dam")
print("="*62)
print("!! Surge chi co o Lyft:")
display(df.groupby("cab_type").agg(so_cuoc=("price","size"),
        surge_max=("surge_multiplier","max"), so_surge=("is_surge","sum")))

**2. Phân bố các trường**

In [ ]:
NUM=["price","distance","price_per_mile","hour_local","temperature","apparentTemperature",
     "precipIntensity","precipProbability","humidity","windSpeed","windGust","visibility",
     "dewPoint","pressure","cloudCover","uvIndex","ozone","moonPhase"]
NUM=[c for c in NUM if c in df.columns]
ncols=4; nrows=math.ceil(len(NUM)/ncols)
fig,axes=plt.subplots(nrows,ncols,figsize=(ncols*3.4,nrows*2.5))
axes=np.atleast_1d(axes).flatten()
for ax in axes[len(NUM):]: ax.axis("off")
for ax,c in zip(axes,NUM):
    x=df[c].dropna()
    ax.hist(x,bins=50,color=BLUE,alpha=.85,zorder=3)
    ax.axvline(x.median(),color=ORANGE,lw=1.5)
    ax.set_title(f"{c}\n(median={x.median():.2f}, skew={x.skew():.1f})",fontsize=8.5)
    ax.tick_params(labelsize=7); ax.grid(axis="x",visible=False)
fig.suptitle("Phan bo cac truong so (vach cam = median)",fontsize=13,fontweight="bold")
fig.tight_layout(rect=[0,0,1,0.98]); plt.show()

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(15,4.2))
axes[0].hist(df.price,bins=70,color=BLUE,alpha=.85)
axes[0].axvline(df.price.median(),color=ORANGE,lw=2,label=f"median={df.price.median():.1f}")
axes[0].set_title("Gia cuoc (USD)"); axes[0].legend(frameon=False)
axes[1].hist(df.distance,bins=70,color=GREEN,alpha=.85)
axes[1].axvline(df.distance.median(),color=ORANGE,lw=2,label=f"median={df.distance.median():.2f}")
axes[1].set_title("Quang duong (dam)"); axes[1].legend(frameon=False)
sv=dfL.surge_multiplier.value_counts().sort_index()
axes[2].bar(sv.index.astype(str),sv.values,color=RED,alpha=.85,width=.6)
axes[2].set_yscale("log"); axes[2].set_title(f"Surge — CHI LYFT ({dfL.is_surge.mean()*100:.2f}% co surge)")
for i,(k,v) in enumerate(sv.items()): axes[2].text(i,v,f"{v:,}",ha="center",va="bottom",fontsize=7.5)
for a in axes: a.grid(axis="x",visible=False)
fig.tight_layout(); plt.show()

**3. Tương quan giữa các trường**

In [ ]:
COR=NUM+["surge_multiplier","is_weekend"]
Cm=df[COR].corr()
fig,ax=plt.subplots(figsize=(11,9.5))
im=ax.imshow(Cm,cmap="RdBu_r",vmin=-1,vmax=1)
ax.set_xticks(range(len(COR))); ax.set_xticklabels(COR,rotation=90,fontsize=8)
ax.set_yticks(range(len(COR))); ax.set_yticklabels(COR,fontsize=8)
for i in range(len(COR)):
    for j in range(len(COR)):
        v=Cm.iloc[i,j]
        if abs(v)>0.45 and i!=j:
            ax.text(j,i,f"{v:.2f}",ha="center",va="center",fontsize=6.5,
                    color="white" if abs(v)>0.7 else "#222")
ax.set_title("Ma tran tuong quan Pearson",fontsize=13,fontweight="bold")
fig.colorbar(im,fraction=0.046,pad=0.04).set_label("r"); ax.grid(False)
fig.tight_layout(); plt.show()

cm=df[COR].corr().abs()
pairs=[(COR[i],COR[j],round(cm.iloc[i,j],3)) for i in range(len(COR))
       for j in range(i+1,len(COR)) if cm.iloc[i,j]>0.9]
print("Cap truong TRUNG LAP (|r|>0.9):")
display(pd.DataFrame(sorted(pairs,key=lambda x:-x[2]),columns=["Trường A","Trường B","|r|"]))

**4. ⭐ Yếu tố nào quyết định GIÁ?**
So sánh tương quan với **`price`** (giá thô) và **`price_per_mile`** (đã bỏ ảnh hưởng quãng đường).

In [ ]:
rows=[]
for c in NUM:
    if c in ("price","price_per_mile"): continue
    m1=df[c].notna()&df.price.notna(); m2=df[c].notna()&df.price_per_mile.notna()
    rows.append((c, np.corrcoef(df[c][m1],df.price[m1])[0,1],
                    np.corrcoef(df[c][m2],df.price_per_mile[m2])[0,1]))
t=pd.DataFrame(rows,columns=["feature","r_với_price","r_với_price_per_mile"])
t["_a"]=t.r_với_price.abs()
display(t.sort_values("_a",ascending=False).drop(columns="_a").round(4).reset_index(drop=True))

In [ ]:
def corr_ratio(cats,y):
    cats=cats.astype(str); ybar=y.mean(); sst=((y-ybar)**2).sum()
    ssb=sum(len(g)*(g.mean()-ybar)**2 for _,g in y.groupby(cats))
    return np.sqrt(ssb/sst) if sst>0 else 0
CAT=["name","cab_type","source","destination","short_summary"]
e=[(c,corr_ratio(df[c],df.price),corr_ratio(df[c],df.price_per_mile.fillna(df.price_per_mile.median())))
   for c in CAT]
print("Suc manh truong PHAN LOAI (eta, 0-1):")
display(pd.DataFrame(e,columns=["feature","eta_với_price","eta_với_price_per_mile"])
          .sort_values("eta_với_price",ascending=False).round(4).reset_index(drop=True))

In [ ]:
s=df.sample(min(40000,len(df)),random_state=0)
fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].scatter(s.distance,s.price,s=4,alpha=.06,color=BLUE,edgecolors="none")
z=np.polyfit(df.distance,df.price,1); xs=np.linspace(df.distance.min(),df.distance.max(),100)
axes[0].plot(xs,np.polyval(z,xs),color=ORANGE,lw=2,label=f"y={z[0]:.2f}x+{z[1]:.2f}")
axes[0].set_xlabel("Quang duong (dam)"); axes[0].set_ylabel("Gia (USD)")
axes[0].set_title(f"Gia vs Quang duong (r={np.corrcoef(df.distance,df.price)[0,1]:.3f})")
axes[0].legend(frameon=False)
order=df.groupby("name").price.median().sort_values().index
axes[1].boxplot([df.loc[df.name==n,"price"].values for n in order],vert=False,showfliers=False,
                patch_artist=True,boxprops=dict(facecolor=BLUE,alpha=.6))
axes[1].set_yticklabels(order,fontsize=8); axes[1].set_xlabel("Gia (USD)")
axes[1].set_title("Gia theo loai dich vu"); axes[1].grid(axis="y",visible=False)
fig.tight_layout(); plt.show()

**5. Thống kê theo thời gian**

In [ ]:
daily=df.groupby("date_local").agg(so_cuoc=("price","size"),gia_TB=("price","mean")).reset_index()
dailyL=dfL.groupby("date_local").agg(ty_le_surge=("is_surge","mean")).reset_index()
fig,axes=plt.subplots(3,1,figsize=(13,8.5),sharex=True)
axes[0].bar(daily.date_local,daily.so_cuoc,color=BLUE,alpha=.85,width=.7); axes[0].set_ylabel("So cuoc")
axes[0].set_title("Thong ke theo ngay",fontweight="bold")
axes[1].plot(daily.date_local,daily.gia_TB,marker="o",color=GREEN); axes[1].set_ylabel("Gia TB (USD)")
axes[2].plot(dailyL.date_local,dailyL.ty_le_surge*100,marker="o",color=RED)
axes[2].set_ylabel("% surge (Lyft)"); axes[2].set_xlabel("Ngay")
for a in axes: a.grid(axis="x",visible=False)
fig.tight_layout(); plt.show()
print("Ngay thieu du lieu: 05-08/12 va 10-11/12")

**6. ⭐ Xác định GIỜ CAO ĐIỂM**
> Dùng **`dfL` (chỉ Lyft)** vì Uber không có dữ liệu surge.

In [ ]:
hourly=dfL.groupby("hour_local").agg(
    so_cuoc=("price","size"), ty_le_surge=("is_surge","mean"),
    multiplier_TB=("surge_multiplier","mean"), gia_TB=("price","mean"),
    gia_moi_dam=("price_per_mile","median")).reset_index()
thr=hourly.ty_le_surge.quantile(0.75)
PEAK_HOURS=sorted(hourly.loc[hourly.ty_le_surge>=thr,"hour_local"].tolist())
print(f"Nguong (phan vi 75 cua ty le surge theo gio): {thr*100:.2f}%")
print(f">> GIO CAO DIEM: {PEAK_HOURS}")
display(hourly.round(4))

In [ ]:
fig,axes=plt.subplots(2,2,figsize=(14,8))
a=axes[0,0]; a.bar(hourly.hour_local,hourly.so_cuoc,color=BLUE,alpha=.85)
a.set_title("Nhu cau: so cuoc theo gio (Lyft)"); a.set_xlabel("Gio")
a=axes[0,1]
a.bar(hourly.hour_local,hourly.ty_le_surge*100,
      color=[RED if h in PEAK_HOURS else MUT for h in hourly.hour_local],alpha=.9)
a.axhline(thr*100,color=ORANGE,ls="--",lw=1.5,label=f"nguong {thr*100:.2f}%")
a.set_title("Ty le surge theo gio (do = CAO DIEM)"); a.set_xlabel("Gio"); a.legend(frameon=False)
a=axes[1,0]; a.plot(hourly.hour_local,hourly.multiplier_TB,marker="o",color=RED)
a.set_title("Multiplier trung binh theo gio"); a.set_xlabel("Gio")
a=axes[1,1]; a.plot(hourly.hour_local,hourly.gia_moi_dam,marker="o",color=GREEN)
a.set_title("Gia moi dam theo gio"); a.set_xlabel("Gio"); a.set_ylabel("USD/dam")
for ax_ in axes.flat: ax_.grid(axis="x",visible=False)
fig.tight_layout(); plt.show()

In [ ]:
DAYS=["T2","T3","T4","T5","T6","T7","CN"]
pv=dfL.pivot_table(index="weekday_local",columns="hour_local",values="is_surge",aggfunc="mean")*100
fig,ax=plt.subplots(figsize=(14,3.6))
im=ax.imshow(pv,cmap="Reds",aspect="auto")
ax.set_xticks(range(pv.shape[1])); ax.set_xticklabels(pv.columns)
ax.set_yticks(range(len(pv))); ax.set_yticklabels([DAYS[i] for i in pv.index])
ax.set_xlabel("Gio"); ax.set_title("Ty le surge (%) theo Gio x Thu — Lyft",fontweight="bold")
fig.colorbar(im,fraction=0.025).set_label("% surge"); ax.grid(False)
fig.tight_layout(); plt.show()

**7. ⭐ Giờ cao điểm ảnh hưởng đến giá thế nào?**

In [ ]:
df["is_peak"]  = df.hour_local.isin(PEAK_HOURS).astype(int)
dfL["is_peak"] = dfL.hour_local.isin(PEAK_HOURS).astype(int)

def cmp(data,col,label):
    a=data.loc[data.is_peak==1,col].dropna(); b=data.loc[data.is_peak==0,col].dropna()
    _,pv=stats.mannwhitneyu(a,b,alternative="two-sided")
    return {"Chỉ số":label,"Cao điểm":round(a.mean(),4),"Thấp điểm":round(b.mean(),4),
            "Chênh %":round((a.mean()-b.mean())/b.mean()*100,2),
            "p-value":f"{pv:.1e}","Ý nghĩa TK":"CÓ" if pv<0.05 else "KHÔNG"}

res=pd.DataFrame([cmp(df,"price","Giá thô (cả 2 hãng)"),
                  cmp(df,"price_per_mile","Giá/dặm (cả 2 hãng)"),
                  cmp(dfL,"surge_multiplier","Surge multiplier (Lyft)"),
                  cmp(dfL,"is_surge","Tỷ lệ có surge (Lyft)"),
                  cmp(dfL,"price_per_mile","Giá/dặm (Lyft)")])
print(f"Gio cao diem: {PEAK_HOURS}")
display(res)
print("!! Chu y: n rat lon nen p-value luon 'co y nghia' -> hay doc cot 'Chenh %'")

In [ ]:
# So sanh CO KIEM SOAT loai dich vu
ctrl=dfL.groupby(["name","is_peak"]).agg(gia_moi_dam=("price_per_mile","median"),
                                          ty_le_surge=("is_surge","mean")).unstack()
out=pd.DataFrame({
 "Giá/dặm thấp điểm": ctrl[("gia_moi_dam",0)].round(3),
 "Giá/dặm cao điểm" : ctrl[("gia_moi_dam",1)].round(3),
 "% surge thấp điểm": (ctrl[("ty_le_surge",0)]*100).round(2),
 "% surge cao điểm" : (ctrl[("ty_le_surge",1)]*100).round(2)})
out["Chênh surge (điểm %)"]=(out["% surge cao điểm"]-out["% surge thấp điểm"]).round(2)
print("Trong TUNG loai dich vu Lyft:")
display(out.sort_values("Chênh surge (điểm %)",ascending=False))

**8. ⭐ Thời tiết ảnh hưởng thế nào?**

In [ ]:
w=dfL.groupby("short_summary").agg(so_cuoc=("price","size"),ty_le_surge=("is_surge","mean"),
      multiplier_TB=("surge_multiplier","mean"),gia_moi_dam=("price_per_mile","median"))
w=w[w.so_cuoc>=300].sort_values("ty_le_surge",ascending=False)
w["ty_le_surge"]=(w.ty_le_surge*100).round(2)
display(w.round(3))

fig,axes=plt.subplots(1,2,figsize=(14,4.6))
axes[0].barh(range(len(w)),w.ty_le_surge.values[::-1],color=RED,alpha=.85)
axes[0].set_yticks(range(len(w))); axes[0].set_yticklabels(w.index[::-1],fontsize=8)
axes[0].set_xlabel("% cuoc co surge"); axes[0].set_title("Surge theo thoi tiet (Lyft)")
axes[1].barh(range(len(w)),w.gia_moi_dam.values[::-1],color=GREEN,alpha=.85)
axes[1].set_yticks(range(len(w))); axes[1].set_yticklabels(w.index[::-1],fontsize=8)
axes[1].set_xlabel("USD/dam"); axes[1].set_title("Gia moi dam theo thoi tiet")
for a in axes: a.grid(axis="y",visible=False)
fig.tight_layout(); plt.show()

In [ ]:
# Muc do mua -> surge
dfL["nhom_mua"]=pd.cut(dfL.precipIntensity,[-0.001,0.0001,0.01,0.05,10],
                        labels=["Không mưa","Mưa rất nhẹ","Mưa nhẹ","Mưa vừa/to"])
r=dfL.groupby("nhom_mua").agg(so_cuoc=("price","size"),ty_le_surge=("is_surge","mean"),
                               gia_moi_dam=("price_per_mile","median"))
r["ty_le_surge"]=(r.ty_le_surge*100).round(2)
display(r.round(3))
base=r.ty_le_surge.iloc[0]
print(f"\nMua vua/to lam ty le surge doi: {(r.ty_le_surge.iloc[-1]-base)/base*100:+.1f}% so voi khong mua")

**9. ⭐ Vị trí ảnh hưởng thế nào? (trung tâm có đắt hơn?)**

In [ ]:
# Khoang cach tu tam Boston toi tung khu vuc
LAT0,LON0 = 42.3601,-71.0589      # downtown Boston
def haversine(la1,lo1,la2,lo2):
    R=3958.8; p1,p2=np.radians(la1),np.radians(la2)
    dp,dl=np.radians(la2-la1),np.radians(lo2-lo1)
    a=np.sin(dp/2)**2+np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

loc=df.groupby("source").agg(lat=("latitude","median"),lon=("longitude","median"),
        so_cuoc=("price","size"),gia_moi_dam=("price_per_mile","median"),
        quang_duong_TB=("distance","mean"))
loc["km_tu_trung_tam"]=haversine(LAT0,LON0,loc.lat,loc.lon).round(2)
sg=dfL.groupby("source").agg(ty_le_surge=("is_surge","mean"))
loc["ty_le_surge_%"]=(sg.ty_le_surge*100).round(2)
loc=loc.sort_values("gia_moi_dam",ascending=False)
display(loc[["so_cuoc","km_tu_trung_tam","quang_duong_TB","gia_moi_dam","ty_le_surge_%"]].round(3))

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
o=loc.sort_values("gia_moi_dam")
axes[0].barh(range(len(o)),o.gia_moi_dam,color=GREEN,alpha=.85)
axes[0].set_yticks(range(len(o))); axes[0].set_yticklabels(o.index,fontsize=8)
axes[0].set_xlabel("USD/dam"); axes[0].set_title("Gia moi dam theo khu vuc don")
axes[1].scatter(loc.km_tu_trung_tam,loc.gia_moi_dam,s=70,color=BLUE,alpha=.8)
for k,v in loc.iterrows():
    axes[1].annotate(k,(v.km_tu_trung_tam,v.gia_moi_dam),fontsize=7,
                     xytext=(3,3),textcoords="offset points")
rr=np.corrcoef(loc.km_tu_trung_tam,loc.gia_moi_dam)[0,1]
axes[1].set_xlabel("Khoang cach tu trung tam Boston (dam)"); axes[1].set_ylabel("USD/dam")
axes[1].set_title(f"Cang xa trung tam cang re? (r={rr:.3f})")
axes[0].grid(axis="y",visible=False)
fig.tight_layout(); plt.show()
print(f"Tuong quan khoang-cach-tu-trung-tam vs gia/dam: r = {rr:.3f}")

**⚠️ Cảnh báo: `price_per_mile` bị quãng đường làm nhiễu**

Phí mở cửa được chia cho số dặm, nên **chuyến càng ngắn thì giá/dặm càng cao**.
Khu vực nào có chuyến ngắn sẽ *trông như* đắt đỏ dù đơn giá không hề cao hơn.

→ Không được kết luận "khu X đắt nhất" chỉ dựa vào `price_per_mile`.
Phải **so sánh trong cùng dải quãng đường**.

In [ ]:
# Chung minh nhieu: tuong quan giua quang duong TB cua khu va gia/dam
_l = df.groupby("source").agg(qd=("distance","mean"), ppm=("price_per_mile","median"))
_r = np.corrcoef(_l.qd, _l.ppm)[0,1]
print(f"Tuong quan (quang duong TB cua khu) vs (gia/dam): r = {_r:.3f}")
print("=> |r| rat cao -> xep hang khu theo gia/dam la KET LUAN GIA.\n")

# So sanh CO KIEM SOAT: gia trung vi trong TUNG dai quang duong
df["dai_qd"] = pd.cut(df.distance, [0,1,2,3,4,10],
                      labels=["<1 dam","1-2","2-3","3-4",">4"])
pv = df.pivot_table(index="source", columns="dai_qd", values="price", aggfunc="median")

# Chi so premium: gia cua khu so voi trung binh cung dai quang duong
prem = (pv / pv.mean(axis=0) - 1) * 100
prem["PREMIUM_TB_%"] = prem.mean(axis=1).round(2)
prem = prem.sort_values("PREMIUM_TB_%", ascending=False).round(2)
print("Gia cao/thap hon bao nhieu % so voi cac khu khac, TRONG CUNG dai quang duong:")
display(prem)

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
d = prem["PREMIUM_TB_%"].sort_values()
cols = [RED if v>0 else BLUE for v in d.values]
ax.barh(range(len(d)), d.values, color=cols, alpha=.85, zorder=3)
ax.set_yticks(range(len(d))); ax.set_yticklabels(d.index, fontsize=9)
ax.axvline(0, color="#222", lw=1)
ax.set_xlabel("% chenh lech gia so voi trung binh (da kiem soat quang duong)")
ax.set_title("Premium theo khu vuc — DA LOAI BO anh huong quang duong", fontweight="bold")
ax.grid(axis="y", visible=False); fig.tight_layout(); plt.show()

print(f"Khu dat nhat that su: {d.index[-1]}  ({d.iloc[-1]:+.2f}%)")
print(f"Khu re nhat that su : {d.index[0]}  ({d.iloc[0]:+.2f}%)")
print(f"Bien do that su chi  : {d.max()-d.min():.1f} diem %  "
      f"(so voi 2x neu nhin gia/dam -> nhieu hon nhieu)")

**10. Feature selection**

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

FN=[c for c in NUM if c not in ("price","price_per_mile")]+["is_weekend","weekday_local"]
FC=["cab_type","name","source","destination","short_summary"]
F=FC+FN

def imp_for(data,target,clf=False,title=""):
    X=data[F].copy()
    for c in FC: X[c]=X[c].astype("category")
    y=data[target]
    Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,random_state=42)
    M=HistGradientBoostingClassifier if clf else HistGradientBoostingRegressor
    m=M(max_iter=200,learning_rate=.05,categorical_features=FC,random_state=42).fit(Xtr,ytr)
    print(f"{title} — diem tren test: {m.score(Xte,yte):.4f}")
    pi=permutation_importance(m,Xte,yte,n_repeats=3,random_state=0)
    d=pd.DataFrame({"feature":F,"importance":pi.importances_mean}).sort_values("importance",ascending=False)
    fig,ax=plt.subplots(figsize=(8,5)); dd=d.head(12)[::-1]
    ax.barh(range(len(dd)),np.clip(dd.importance,0,None),color=ORANGE,zorder=3)
    ax.set_yticks(range(len(dd))); ax.set_yticklabels(dd.feature,fontsize=9)
    ax.set_title(title,fontweight="bold"); ax.grid(axis="y",visible=False)
    fig.tight_layout(); plt.show()
    return d.reset_index(drop=True)

imp_price = imp_for(df,"price",False,"Permutation importance — GIA")
display(imp_price.head(10).round(4))

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

X = dfL[F].copy()
for c in FC: X[c] = X[c].astype("category")
y = dfL["is_surge"]
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
m = HistGradientBoostingClassifier(max_iter=200,learning_rate=.05,
        categorical_features=FC,random_state=42).fit(Xtr,ytr)
proba = m.predict_proba(Xte)[:,1]

base_acc = 1 - yte.mean()          # doan "khong surge" cho tat ca
acc      = m.score(Xte,yte)
auc      = roc_auc_score(yte, proba)
ap       = average_precision_score(yte, proba)

print("="*62); print("DANH GIA MODEL DU DOAN CO SURGE (Lyft)"); print("="*62)
print(f"Ty le co surge trong test      : {yte.mean()*100:.2f}%  (rat mat can bang)")
print(f"Accuracy khi doan bua 'khong'  : {base_acc:.4f}   <- nguong tam thuong")
print(f"Accuracy cua model             : {acc:.4f}   (hon nguong {acc-base_acc:+.4f})")
print(f"ROC-AUC                        : {auc:.4f}   (0.5 = doan bua)")
print(f"PR-AUC (average precision)     : {ap:.4f}   (nen so voi {yte.mean():.4f})")
print()
if acc - base_acc < 0.01:
    print(">> KET LUAN: Accuracy GAN NHU KHONG hon doan bua -> accuracy la chi so SAI")
    print("   cho bai toan mat can bang. Phai doc ROC-AUC / PR-AUC.")
if auc > 0.7:
    print(f">> Nhung ROC-AUC = {auc:.3f} cho thay model VAN phan biet duoc phan nao.")
    print("   -> Van con tin hieu, chi la nguong quyet dinh 0.5 khong phu hop.")

pi = permutation_importance(m, Xte, yte, n_repeats=3, random_state=0,
                            scoring="roc_auc")
imp_surge = pd.DataFrame({"feature":F,"importance":pi.importances_mean}) \
              .sort_values("importance",ascending=False).reset_index(drop=True)
fig,ax=plt.subplots(figsize=(8,5)); dd=imp_surge.head(12)[::-1]
ax.barh(range(len(dd)),np.clip(dd.importance,0,None),color=PURPLE,zorder=3)
ax.set_yticks(range(len(dd))); ax.set_yticklabels(dd.feature,fontsize=9)
ax.set_title("Permutation importance — CO SURGE (do bang ROC-AUC)",fontweight="bold")
ax.grid(axis="y",visible=False); fig.tight_layout(); plt.show()
display(imp_surge.head(10).round(5))

**11. 📌 Tổng kết — số liệu để đưa vào report**

In [ ]:
r=res.set_index("Chỉ số")
print("="*72); print("KET QUA CHINH — TUAN 1"); print("="*72)
print(f"Du lieu: {len(df):,} chuyen | {df.date_local.nunique()} ngay | Lyft {len(dfL):,} (surge {dfL.is_surge.sum():,})")
print()
print("1) YEU TO QUYET DINH GIA")
print(f"   - Loai dich vu (eta voi gia): {corr_ratio(df['name'],df['price']):.3f}  <- manh nhat")
print(f"   - Quang duong (r voi gia)   : {np.corrcoef(df.distance,df.price)[0,1]:.3f}")
print(f"   - Thoi tiet                 : |r| < 0.05 (khong dang ke)")
print()
print("2) GIO CAO DIEM =", PEAK_HOURS)
print(f"   - Gia tho        : {r.loc['Giá thô (cả 2 hãng)','Chênh %']:+.2f}%")
print(f"   - Gia/dam        : {r.loc['Giá/dặm (cả 2 hãng)','Chênh %']:+.2f}%")
print(f"   - Surge (Lyft)   : {r.loc['Surge multiplier (Lyft)','Chênh %']:+.2f}%")
print(f"   - Ty le surge    : {r.loc['Tỷ lệ có surge (Lyft)','Chênh %']:+.2f}%")
print()
print("3) VI TRI")
print(f"   - Tuong quan xa-trung-tam vs gia/dam: r = {rr:.3f}")
print(f"   - Khu dat nhat : {loc.index[0]} ({loc.gia_moi_dam.iloc[0]:.2f} USD/dam)")
print(f"   - Khu re nhat  : {loc.index[-1]} ({loc.gia_moi_dam.iloc[-1]:.2f} USD/dam)")
print()
print("4) HAN CHE DU LIEU")
print("   - Uber khong co surge (330,568 dong = 1.0) -> chi phan tich duoc tren Lyft")
print("   - Khong co tin hieu cung-cau, khong co ket qua dat xe")
print("   - Chi 18 ngay, 1 thanh pho, mua dong 2018")

---
**💡 Gợi ý punchline cho report**
> *"Giá thô được quyết định gần như hoàn toàn bởi **loại dịch vụ + quãng đường**; thời tiết/thời gian/vị trí hầu như không đổi được giá thô — chúng chỉ tác động qua **surge multiplier**. Do đó phải **tách 2 target** (`price` và `multiplier`) mô hình hoá riêng, và multiplier chỉ học được trên dữ liệu Lyft."*
